# 06 — MVTec: revision diagnostics (confirmation-gap decomposition, sensitivity)

Produces the MVTec side of three analyses reported in Sections 4.7-4.9:

- **Part A** — per-query compute timing (component speed-up feeding Table 6; see the
  in-cell note about the energy/CO2 content removed from this release).
- **Part B** — diagnoses the confirmation gap: re-runs each of the 24 confirmed candidates once
  with identical seeds, capturing raw per-image anomaly scores (no retraining), then computes
  the gap-vs-distance correlation, a review-budget threshold sweep, threshold-transfer loss, and
  the pooled-vs-per-category AUROC breakdown that explains the near-chance pooled AUROC
  (**Table 4**, **Figure 3**).
- **Part C** — repeats the DiCE/LogTopK generator comparison at 5 percentile-based factual
  anchors (Section 4.8 sensitivity sweep), surrogate-level only, no additional retraining.

**Public-release fix:** the evaluation-CSV lookup in the setup cell now searches the dataset's
own `mvtec_counterfactuals_evaluation/` folder by filename suffix instead of an exact-name
match, matching the (already-correct) pattern used by the Proditec twin notebook — an
exact-name search can silently pick up a wrong-dataset file of the same name in a shared
working tree.


# MVTec (4-class) — Revision analysis (FGCS special-issue review)

This notebook produces the additional evidence and figures requested by the review, on top of the
existing pipeline outputs (surrogate dataset from notebook 2, surrogate model from notebook 3,
all-method counterfactuals from notebook 4, and the real PatchCore confirmation table from notebook 5).

| Part | Review comment | What is produced |
|---|---|---|
| A | **M1** — sustainability / compute-efficiency framing for the SI scope | Per-query cost of surrogate vs. real PatchCore retraining, energy and CO2 estimates, amortisation of the HPO log |
| B | **M2** — diagnose the confirmation gap (near-chance MVTec AUROC) | Gap-vs-distance analysis, inspection of failing configurations, score capture for all confirmed candidates, review-budget sweep, threshold-transfer loss, per-category breakdown (MVTec) |
| C | **M4** — single-factual-configuration limitation | Multi-factual sensitivity: the generator comparison repeated across the percentile-based factual anchors (50/70/85/95/best), with optional real-model confirmation |

All figures are saved in `figures_revision/` (pdf + svg + png) and all tables in `tables_revision/`
(csv + tex). A manifest mapping outputs to review comments is written at the end.

Run order: notebooks 1-5 must have been executed at least once for this dataset so that the
surrogate dataset, the CatBoost surrogate, the all-method CF table, and the PatchCore
confirmation CSV exist on disk.

In [ ]:
# ============================================================
# 0) Setup, robust path resolution, output directories
# ============================================================

from pathlib import Path
import json
import time
import zlib
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 180)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CWD = Path.cwd()
print("Current working directory:")
print(CWD)

# ------------------------------------------------------------
# Expected filenames from previous MVTec (4-class) notebooks
# ------------------------------------------------------------

DATA_FILENAME = "mvtec_4class_patchcore_surrogate_dataset.csv"
MODEL_FILENAME = "mvtec_surrogate_catboost.cbm"
META_FILENAME = "mvtec_surrogate_metadata.json"

CF_SUBDIR = "mvtec_counterfactuals"
ALL_METHODS_CFS_FILENAME = "mvtec_all_methods_cfs.csv"
CFS_META_FILENAME = "mvtec_all_methods_cfs_metadata.json"

EVAL_CSV_FILENAME = "mvtec_all_methods_cfs_patchcore_evaluation.csv"
SPLITS_SUBDIR = "mvtec_splits"


def find_files(filename):
    # Search CWD and its parent recursively; return unique resolved hits.
    hits = []
    for base in [CWD, CWD.parent]:
        try:
            hits.extend(base.rglob(filename))
        except Exception:
            pass
    out = []
    seen = set()
    for p in hits:
        p = p.resolve()
        if p not in seen:
            seen.add(p)
            out.append(p)
    return out


data_hits = find_files(DATA_FILENAME)
if not data_hits:
    raise FileNotFoundError(
        f"Could not find the surrogate-ready dataset ({DATA_FILENAME}). "
        "Run the preprocessing notebook first."
    )

CSV_PATH = data_hits[0]
ROOT = CSV_PATH.parent.parent

ART_DIR = ROOT / "surrogate_pkl_cfs_metadata"
CFS_DIR = ART_DIR / CF_SUBDIR
MODEL_PATH = ART_DIR / MODEL_FILENAME
META_PATH = ART_DIR / META_FILENAME
CFS_PATH = CFS_DIR / ALL_METHODS_CFS_FILENAME
CFS_META_PATH = CFS_DIR / CFS_META_FILENAME
SPLITS_DIR = ROOT / SPLITS_SUBDIR

# Public-release fix: search the dataset-s own evaluation folder by suffix
# instead of matching EVAL_CSV_FILENAME exactly. An exact-name search can
# collide with a same-named file for a DIFFERENT dataset elsewhere in a
# shared working tree (this happened in practice on the Proditec side; see
# 7th_proditec_revision_analysis.ipynb, which already used this pattern).
EVAL_SUBDIR = ART_DIR / f"{CF_SUBDIR}_evaluation"

def find_eval_csv():
    candidates = []
    if EVAL_SUBDIR.exists():
        candidates.extend(sorted(EVAL_SUBDIR.glob("*patchcore_evaluation.csv")))
    if not candidates:
        candidates.extend(find_files(EVAL_CSV_FILENAME))
    candidates = [
        p for p in candidates
        if "_summary" not in p.stem and "_metadata" not in p.stem
    ]
    return candidates[0] if candidates else None

EVAL_CSV_PATH = find_eval_csv()

# Revision-specific output directories.
FIG_DIR = ROOT / "figures_revision"
TABLE_DIR = ROOT / "tables_revision"
REV_DIR = ART_DIR / "mvtec_revision_analysis"
SCORES_DIR = REV_DIR / "captured_scores"

for d in [FIG_DIR, TABLE_DIR, REV_DIR, SCORES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\nResolved paths:")
print("ROOT:", ROOT)
print("CSV_PATH:", CSV_PATH)
print("MODEL_PATH:", MODEL_PATH)
print("CFS_PATH:", CFS_PATH)
print("EVAL_CSV_PATH:", EVAL_CSV_PATH)
print("SPLITS_DIR:", SPLITS_DIR)
print("FIG_DIR:", FIG_DIR)
print("TABLE_DIR:", TABLE_DIR)
print("REV_DIR:", REV_DIR)

print("\nFile checks:")
print("Surrogate dataset:", CSV_PATH.exists())
print("Surrogate model:", MODEL_PATH.exists())
print("All-method CFs:", CFS_PATH.exists())
print("Prior PatchCore evaluation CSV:", EVAL_CSV_PATH is not None and EVAL_CSV_PATH.exists())
print("Splits dir:", SPLITS_DIR.exists())

assert CSV_PATH.is_file(), CSV_PATH
assert MODEL_PATH.is_file(), MODEL_PATH
assert CFS_PATH.is_file(), CFS_PATH

if EVAL_CSV_PATH is None:
    print(
        "\nWARNING: prior PatchCore evaluation CSV not found. "
        "Parts A and B need it; run the confirmation notebook (5) first."
    )

In [ ]:
# ============================================================
# 1) Load schema, surrogate dataset, surrogate model,
#    all-method CFs, and the prior confirmation table
# ============================================================

TARGET = "value"

if META_PATH.exists():
    with open(META_PATH, "r", encoding="utf-8") as f:
        surrogate_meta = json.load(f)
    FEATURES = surrogate_meta["features"]
    CATEGORICAL = surrogate_meta["categorical_features"]
    NUMERIC = surrogate_meta["numeric_features"]
    print("Loaded schema from surrogate metadata.")
else:
    FEATURES = [
        "params_backbone",
        "params_batch_size",
        "params_center_crop_key",
        "params_coreset_sampling_ratio",
        "params_image_size_key",
        "params_layers_key",
        "params_max_patches_per_image",
        "params_num_neighbors",
        "params_pre_trained",
        "params_reduction",
        "params_soft_corruption_level",
        "params_soft_review_budget",
        "params_soft_train_fraction",
    ]
    CATEGORICAL = [
        "params_backbone",
        "params_batch_size",
        "params_center_crop_key",
        "params_image_size_key",
        "params_layers_key",
        "params_max_patches_per_image",
        "params_pre_trained",
        "params_reduction",
        "params_soft_corruption_level",
    ]
    NUMERIC = [c for c in FEATURES if c not in CATEGORICAL]
    print("Surrogate metadata not found. Using fallback schema.")


def coerce_schema(frame):
    x = frame.copy()
    for c in CATEGORICAL:
        if c in x.columns:
            x[c] = x[c].astype(str)
    for c in NUMERIC:
        if c in x.columns:
            x[c] = pd.to_numeric(x[c], errors="coerce")
    return x


def predict_surrogate(model, X):
    Xp = coerce_schema(X[FEATURES])
    return np.asarray(model.predict(Xp), dtype=float)


# Surrogate-ready HPO log.
df = pd.read_csv(CSV_PATH)
missing = [c for c in [TARGET] + FEATURES if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in surrogate dataset: {missing}")

df = coerce_schema(df[[TARGET] + FEATURES])
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET] + FEATURES).reset_index(drop=True)

# CatBoost surrogate.
cb = CatBoostRegressor()
cb.load_model(str(MODEL_PATH))

# All-method CF table (notebook 4 output).
cfs = pd.read_csv(CFS_PATH)
cfs = coerce_schema(cfs)
if "cf_id" not in cfs.columns and "global_cf_id" in cfs.columns:
    cfs["cf_id"] = pd.to_numeric(cfs["global_cf_id"], errors="coerce").astype(int)

# Prior real PatchCore confirmation table (notebook 5 output).
if EVAL_CSV_PATH is not None and EVAL_CSV_PATH.exists():
    eval_prev = pd.read_csv(EVAL_CSV_PATH)
    eval_prev = coerce_schema(eval_prev)
    ok_df = eval_prev[eval_prev["status"] == "ok"].copy().reset_index(drop=True)
else:
    eval_prev = pd.DataFrame()
    ok_df = pd.DataFrame()

print("HPO log:", df.shape)
print("All-method CFs:", cfs.shape)
print("Prior confirmations (ok):", len(ok_df))

if not ok_df.empty:
    keep_preview = [
        c for c in [
            "cf_id", "method", "surrogate_pred", "f1_test", "auroc_test",
            "gap_test_minus_surrogate_pred", "distance_to_factual", "runtime_sec",
        ] if c in ok_df.columns
    ]
    display(ok_df[keep_preview].round(4).head(24))

In [ ]:
# ============================================================
# 2) Paper-ready plotting style and save helpers
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 8,
    "axes.titlesize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SINGLE_COL = (3.35, 2.35)
WIDE_COL = (6.85, 2.65)

saved_figures = []
saved_tables = []

METHOD_ORDER = ["ACFX", "CFNOW", "DiCE"]
METHOD_COLORS = {"ACFX": "tab:blue", "CFNOW": "tab:orange", "DiCE": "tab:green"}
METHOD_MARKERS = {"ACFX": "o", "CFNOW": "s", "DiCE": "^"}


def save_paper_fig(fig, stem):
    paths = []
    for ext in ["pdf", "svg", "png"]:
        out = FIG_DIR / f"{stem}.{ext}"
        fig.savefig(out, bbox_inches="tight")
        paths.append(out)
    plt.close(fig)
    saved_figures.extend(paths)
    print("Saved figure:", FIG_DIR / f"{stem}.pdf")
    return paths


def save_table(table, stem, float_format="%.4f"):
    csv_path = TABLE_DIR / f"{stem}.csv"
    tex_path = TABLE_DIR / f"{stem}.tex"
    table.to_csv(csv_path, index=False)
    table.to_latex(tex_path, index=False, escape=True, float_format=float_format)
    saved_tables.extend([csv_path, tex_path])
    print("Saved table:", csv_path)
    return csv_path, tex_path

## Part A — Compute cost, energy, and sustainability (review comment M1)

The special issue asks for AI that is an *enabler of sustainability*. CF-XHPO has a natural
efficiency argument that the paper did not quantify: once the HPO log exists, every operator
what-if query is answered by a surrogate prediction instead of a full PatchCore retraining.
This part measures both costs on the current hardware, converts them to energy and CO2
estimates, and reports the amortisation of the one-off HPO log over interactive query workloads.

Assumptions (edit below to match your hardware): device power draw during PatchCore retraining
and during surrogate inference, and the carbon intensity of the electricity grid.

> **Public-release note.** This cell now reports measured wall-clock timing only. The energy (kWh) and CO2 (kg) estimates from the reviewer-response draft have been removed because they depended on assumed, unmeasured power constants and the paper (Section 4.10) explicitly states no power telemetry was collected and no energy/CO2 saving is claimed.

In [ ]:
# ============================================================
# A) Per-query timing: surrogate inference vs real PatchCore retraining
#
# NOTE (public-release edit): the original version of this cell also computed
# energy (kWh) and CO2 (kg) estimates from ASSUMED, unmeasured power constants
# (PATCHCORE_POWER_W=300, SURROGATE_POWER_W=65, a fixed grid carbon intensity).
# The paper's Section 4.10 explicitly states that no power telemetry was
# collected and that no energy or CO2 saving is claimed, so those estimates
# are removed here to avoid shipping numbers that contradict the paper. This
# cell keeps only the measured wall-clock timing, which is what Table 6's
# "component speed-up" figure reports.
# ============================================================

if ok_df.empty or "runtime_sec" not in ok_df.columns:
    raise ValueError(
        "Part A needs the prior PatchCore confirmation CSV with runtime_sec. "
        "Run the confirmation notebook (5) first."
    )

# 1) Real-model cost from the confirmed candidates of the paper.
patchcore_runtimes = pd.to_numeric(ok_df["runtime_sec"], errors="coerce").dropna()
patchcore_mean_s = float(patchcore_runtimes.mean())
patchcore_total_s = float(patchcore_runtimes.sum())

# 2) Surrogate query latency on this machine.
warm = predict_surrogate(cb, df.head(8))  # warm-up

single_row = df.sample(1, random_state=SEED)[FEATURES]
N_SINGLE = 200
t0 = time.perf_counter()
for _ in range(N_SINGLE):
    _ = predict_surrogate(cb, single_row)
surrogate_single_s = (time.perf_counter() - t0) / N_SINGLE

batch = df.sample(min(1000, len(df)), random_state=SEED)[FEATURES]
t0 = time.perf_counter()
_ = predict_surrogate(cb, batch)
surrogate_batch_total = time.perf_counter() - t0
surrogate_batch_per_row_s = surrogate_batch_total / len(batch)

speedup_single = patchcore_mean_s / surrogate_single_s

cost_table = pd.DataFrame([
    {
        "query_mode": "Real PatchCore retraining",
        "mean_time_s": patchcore_mean_s,
    },
    {
        "query_mode": "Surrogate what-if query",
        "mean_time_s": surrogate_single_s,
    },
])

print(f"Mean PatchCore retraining time: {patchcore_mean_s:.2f} s "
      f"(n={len(patchcore_runtimes)}, total {patchcore_total_s/3600:.2f} h)")
print(f"Surrogate single-query latency: {surrogate_single_s*1000:.3f} ms "
      f"(batched: {surrogate_batch_per_row_s*1e6:.1f} us/row)")
print(f"Component speed-up of surrogate over retraining: {speedup_single:,.0f}x")
display(cost_table)

save_table(cost_table, "mvtec_compute_cost_per_query", float_format="%.6g")

# 3) Figure: per-query timing on a log scale.
fig, ax = plt.subplots(figsize=SINGLE_COL)
modes = ["PatchCore\nretraining", "Surrogate\nquery"]
vals = [patchcore_mean_s, surrogate_single_s]
bars = ax.bar(modes, vals, width=0.55, color=["tab:red", "tab:green"], alpha=0.85)
ax.set_yscale("log")
ax.set_ylabel("Time per what-if query (s, log scale)")
for b, v in zip(bars, vals):
    label = f"{v:.1f} s" if v >= 1 else f"{v*1000:.1f} ms"
    ax.annotate(label, (b.get_x() + b.get_width() / 2, v),
                ha="center", va="bottom", fontsize=7, xytext=(0, 2),
                textcoords="offset points")
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
ax.set_title(f"MVTec (4-class): {speedup_single:,.0f}x per-query speed-up")
save_paper_fig(fig, "mvtec_compute_cost_per_query")


## Part B — Diagnosing the confirmation gap (review comment M2)

The review asks for a deeper diagnosis of why strong surrogate scores can coexist with weak
confirmed behaviour (near-chance pooled test AUROC on MVTec), beyond attributing it to
"coverage and target shape". This part rules confounds in or out:

1. **Gap vs. distance** — is surrogate extrapolation (mixed-type distance from the factual) the
   dominant error source? (cheap: reuses the existing confirmation table)
2. **Failing-configuration inspection** — are the worst-gap candidates degenerate
   (no pretrained weights, extreme coreset ratios, single feature layer, strong corruption)?
3. **Score capture** — re-run the confirmed candidates once while saving the raw anomaly scores,
   so that thresholding and category effects can be analysed *without further retraining*.
4. **Review-budget sweep and threshold-transfer loss** — how much confirmed F1 depends on the
   fixed budget threshold calibrated on validation scores.
5. **Per-category breakdown** *(MVTec only)* — whether pooling four categories with different
   anomaly-score scales under a single budget threshold explains the near-chance pooled AUROC.

In [ ]:
# ============================================================
# B1) Confirmation gap vs distance to the factual configuration
# ============================================================

from scipy import stats

if ok_df.empty:
    raise ValueError("Part B needs the prior confirmation table. Run notebook 5 first.")

gap_col = "gap_test_minus_surrogate_pred"
dist_col = "distance_to_factual"

b1 = ok_df.dropna(subset=[gap_col, dist_col]).copy()
b1["method"] = b1["method"].astype(str)

corr_rows = []
for name, group in [("all", b1)] + [(m, g) for m, g in b1.groupby("method")]:
    if len(group) < 3:
        continue
    pear_r, pear_p = stats.pearsonr(group[dist_col], group[gap_col])
    spear_r, spear_p = stats.spearmanr(group[dist_col], group[gap_col])
    corr_rows.append({
        "scope": name,
        "n": len(group),
        "pearson_r": pear_r,
        "pearson_p": pear_p,
        "spearman_rho": spear_r,
        "spearman_p": spear_p,
    })

corr_table = pd.DataFrame(corr_rows)
display(corr_table.round(4))
save_table(corr_table, "mvtec_gap_vs_distance_correlations")

fig, ax = plt.subplots(figsize=SINGLE_COL)
for m in [m for m in METHOD_ORDER if m in b1["method"].unique()]:
    g = b1[b1["method"] == m]
    ax.scatter(g[dist_col], g[gap_col], s=26, alpha=0.85,
               marker=METHOD_MARKERS.get(m, "o"),
               color=METHOD_COLORS.get(m), label=f"{m} (n={len(g)})")

x = b1[dist_col].astype(float).to_numpy()
y = b1[gap_col].astype(float).to_numpy()
if len(np.unique(x)) >= 2:
    coef = np.polyfit(x, y, deg=1)
    xl = np.linspace(x.min(), x.max(), 100)
    ax.plot(xl, coef[0] * xl + coef[1], "--", linewidth=0.9, color="gray",
            label="Linear trend")

ax.axhline(0.0, linewidth=0.6, color="black", alpha=0.5)
ax.set_xlabel("Mixed-type distance to factual configuration")
ax.set_ylabel("Confirmation gap\n(confirmed F1 - surrogate F1)")
ax.grid(linewidth=0.4, alpha=0.35)
ax.legend(frameon=False, loc="best")
save_paper_fig(fig, "mvtec_gap_vs_distance")

In [ ]:
# ============================================================
# B2) Inspection of the worst-gap (failing) configurations
# ============================================================

b2 = ok_df.copy()
b2["method"] = b2["method"].astype(str)
b2["abs_gap"] = b2["gap_test_minus_surrogate_pred"].abs()

# Degeneracy flags: configuration properties plausibly linked to weak retrained behaviour.
b2["flag_no_pretrained"] = ~b2["params_pre_trained"].astype(str).str.lower().isin(
    ["true", "1", "yes"]
)
b2["flag_tiny_coreset"] = pd.to_numeric(
    b2["params_coreset_sampling_ratio"], errors="coerce"
) <= 0.003
b2["flag_single_layer"] = b2["params_layers_key"].astype(str).isin(["l2", "l3"])
b2["flag_strong_corruption"] = (
    b2["params_soft_corruption_level"].astype(str) == "strong"
)
b2["flag_low_budget"] = pd.to_numeric(
    b2["params_soft_review_budget"], errors="coerce"
) <= 0.05

flag_cols = [c for c in b2.columns if c.startswith("flag_")]
b2["n_flags"] = b2[flag_cols].sum(axis=1)

inspect_cols = [
    "cf_id", "method", "surrogate_pred", "f1_test", "auroc_test",
    "gap_test_minus_surrogate_pred", "distance_to_factual",
    "params_backbone", "params_pre_trained", "params_layers_key",
    "params_coreset_sampling_ratio", "params_image_size_key",
    "params_soft_review_budget", "params_soft_train_fraction",
    "params_soft_corruption_level", "n_flags",
]
inspect_cols = [c for c in inspect_cols if c in b2.columns]

worst = b2.sort_values("gap_test_minus_surrogate_pred").head(8)
print("Worst-gap candidates (most over-estimated by the surrogate):")
display(worst[inspect_cols].round(4))
save_table(worst[inspect_cols].round(4), "mvtec_worst_gap_candidates", float_format="%.3f")

# Do the flags separate the gap? Mean gap and AUROC with each flag on/off.
flag_rows = []
for fc in flag_cols + ["n_flags"]:
    if fc == "n_flags":
        continue
    on = b2[b2[fc]]
    off = b2[~b2[fc]]
    flag_rows.append({
        "flag": fc.replace("flag_", ""),
        "n_on": len(on),
        "n_off": len(off),
        "mean_gap_on": on["gap_test_minus_surrogate_pred"].mean() if len(on) else np.nan,
        "mean_gap_off": off["gap_test_minus_surrogate_pred"].mean() if len(off) else np.nan,
        "mean_auroc_on": on["auroc_test"].mean() if len(on) else np.nan,
        "mean_auroc_off": off["auroc_test"].mean() if len(off) else np.nan,
    })
flag_table = pd.DataFrame(flag_rows)
display(flag_table.round(4))
save_table(flag_table, "mvtec_degeneracy_flag_summary")

# Gap vs number of flags.
fig, ax = plt.subplots(figsize=SINGLE_COL)
groups = sorted(b2["n_flags"].unique())
data = [b2.loc[b2["n_flags"] == g, "gap_test_minus_surrogate_pred"].values for g in groups]
ax.boxplot(data, labels=[str(int(g)) for g in groups], widths=0.55, showmeans=True)
ax.axhline(0.0, linewidth=0.6, color="black", alpha=0.5)
ax.set_xlabel("Number of degeneracy flags")
ax.set_ylabel("Confirmation gap")
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
save_paper_fig(fig, "mvtec_gap_vs_degeneracy_flags")

### B3 — Score capture (one re-run of the confirmed candidates)

The prior confirmation stored only aggregate metrics. To analyse thresholding and category
effects we need the raw per-image anomaly scores. The cell below re-runs each confirmed
candidate **once** with the same seed offset as in notebook 5 (so the runs are reproductions of
the confirmed results) and caches `y_true`/`y_score` for validation and test to
`captured_scores/cf_<id>.npz`. Cached candidates are skipped, so re-executing this notebook is
cheap after the first pass.

Cost: one full confirmation pass (same as notebook 5). Set `MAX_SCORE_CAPTURE` to a small
number for a smoke test.

In [ ]:
# ============================================================
# B3a) PatchCore stack (identical to notebook 5) + score-capturing runner
# ============================================================

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageFilter
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


def _stable_int_seed(text, seed):
    return (zlib.crc32(text.encode("utf-8")) + seed) & 0xFFFFFFFF


def corrupt_pil(img, level, seed_int):
    if level == "none":
        return img
    rng_local = np.random.default_rng(seed_int)
    out = img
    if level in {"mild", "strong"}:
        radius = 0.6 if level == "mild" else 1.2
        out = out.filter(ImageFilter.GaussianBlur(radius=radius))
    if level == "strong":
        arr = np.asarray(out).astype(np.float32)
        noise = rng_local.normal(0.0, 8.0, size=arr.shape).astype(np.float32)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        out = Image.fromarray(arr)
    return out


class ImagePathDataset(Dataset):
    def __init__(self, frame, preproc, corruption_level="none", apply_corruption=False):
        self.paths = frame["path"].tolist()
        self.labels = frame["label"].to_numpy().astype(int)
        self.preproc = preproc
        self.corruption_level = corruption_level
        self.apply_corruption = apply_corruption

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        img = Image.open(path).convert("RGB")
        if self.apply_corruption and self.corruption_level != "none":
            img = corrupt_pil(img, self.corruption_level, _stable_int_seed(path, SEED))
        x = self.preproc(img)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


def make_preproc(image_size, center_crop_size):
    t = [transforms.Resize(image_size)]
    if center_crop_size is not None:
        t.append(transforms.CenterCrop(center_crop_size))
    t += [
        transforms.ToTensor(),
        transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ]
    return transforms.Compose(t)


def make_loaders(df_train_good_sub, image_size, center_crop_size, batch_size, corruption_level):
    preproc = make_preproc(image_size, center_crop_size)
    train_ds = ImagePathDataset(df_train_good_sub, preproc, corruption_level, True)
    val_ds = ImagePathDataset(df_val, preproc, "none", False)
    test_ds = ImagePathDataset(df_final, preproc, "none", False)
    kwargs = dict(shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())
    return (
        DataLoader(train_ds, batch_size=batch_size, **kwargs),
        DataLoader(val_ds, batch_size=batch_size, **kwargs),
        DataLoader(test_ds, batch_size=batch_size, **kwargs),
    )


def build_backbone(backbone, pre_trained):
    if backbone == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet34":
        m = models.resnet34(weights=models.ResNet34_Weights.DEFAULT if pre_trained else None)
    elif backbone == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pre_trained else None)
    elif backbone == "wide_resnet50_2":
        m = models.wide_resnet50_2(
            weights=models.Wide_ResNet50_2_Weights.DEFAULT if pre_trained else None
        )
    else:
        raise ValueError(f"Unsupported backbone: {backbone}")
    m.eval()
    for p in m.parameters():
        p.requires_grad = False
    return m


class FeatureExtractor(torch.nn.Module):
    def __init__(self, backbone, layers, pre_trained):
        super().__init__()
        self.backbone = build_backbone(backbone, pre_trained)
        self.layers = layers
        self._features = {}
        named = dict(self.backbone.named_modules())
        for name in layers:
            if name not in named:
                raise ValueError(f"Layer '{name}' not found in backbone '{backbone}'")
            named[name].register_forward_hook(self._hook(name))

    def _hook(self, name):
        def fn(module, inp, out):
            self._features[name] = out
        return fn

    @torch.no_grad()
    def forward(self, x):
        self._features = {}
        _ = self.backbone(x)
        return [self._features[name] for name in self.layers]


def to_patch_embeddings(feat_maps, max_patches_per_image=None, seed=0):
    H = max(f.shape[-2] for f in feat_maps)
    W = max(f.shape[-1] for f in feat_maps)
    resized = []
    for f in feat_maps:
        if (f.shape[-2], f.shape[-1]) != (H, W):
            f = torch.nn.functional.interpolate(f, size=(H, W), mode="bilinear",
                                                align_corners=False)
        resized.append(f)
    f_cat = torch.cat(resized, dim=1)
    B, D, _, _ = f_cat.shape
    f_flat = f_cat.flatten(2).transpose(1, 2).contiguous()
    P = f_flat.shape[1]
    if max_patches_per_image is not None and max_patches_per_image < P:
        g = torch.Generator(device=f_flat.device)
        g.manual_seed(seed)
        idx = torch.randperm(P, generator=g, device=f_flat.device)[:max_patches_per_image]
        f_flat = f_flat[:, idx, :]
    return f_flat.reshape(-1, D)


def coreset_sample(embeddings, ratio, seed):
    n = embeddings.shape[0]
    m = max(1, int(n * ratio))
    rng_local = np.random.default_rng(seed)
    idx = rng_local.choice(n, size=m, replace=False)
    return embeddings[idx]


@torch.no_grad()
def fit_memory_bank(extractor, train_loader, coreset_sampling_ratio, max_patches_per_image):
    extractor = extractor.to(DEVICE).eval()
    kept = []
    for x, _ in train_loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)
        patches = to_patch_embeddings(feats, max_patches_per_image, SEED)
        kept.append(patches.detach().cpu().numpy())
        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    emb = np.vstack(kept)
    return coreset_sample(emb, coreset_sampling_ratio, SEED)


@torch.no_grad()
def score_images(extractor, loader, memory_bank, num_neighbors, reduction,
                 max_patches_per_image):
    extractor = extractor.to(DEVICE).eval()
    nn = NearestNeighbors(n_neighbors=num_neighbors, algorithm="auto")
    nn.fit(memory_bank)
    y_true_list, y_score_list = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        feats = extractor(x)
        patches = (
            to_patch_embeddings(feats, max_patches_per_image, SEED)
            .detach().cpu().numpy()
        )
        dists, _ = nn.kneighbors(patches, return_distance=True)
        patch_scores = dists.mean(axis=1)
        B = y.shape[0]
        P = patch_scores.shape[0] // B
        patch_scores = patch_scores.reshape(B, P)
        if reduction == "max":
            img_scores = patch_scores.max(axis=1)
        elif reduction == "mean":
            img_scores = patch_scores.mean(axis=1)
        else:
            raise ValueError(f"Unsupported reduction: {reduction}")
        y_true_list.append(y.numpy())
        y_score_list.append(img_scores)
        del x, feats, patches
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return np.concatenate(y_true_list), np.concatenate(y_score_list)


def threshold_from_review_budget(y_score, budget):
    assert 0.0 < budget < 1.0
    n = y_score.shape[0]
    k = max(1, int(np.ceil(budget * n)))
    return float(np.partition(y_score, -k)[-k])


LAYERS_MAP = {
    "l2_l3": ("layer2", "layer3"),
    "l1_l2_l3": ("layer1", "layer2", "layer3"),
    "l3": ("layer3",),
    "l2": ("layer2",),
}

IMAGE_SIZE_MAP = {"224": (224, 224), "256": (256, 256), "320": (320, 320)}


def _as_bool(x):
    if isinstance(x, bool):
        return x
    return str(x).strip().lower() in {"1", "true", "yes", "y", "t"}


def decode_patchcore_config(cfg):
    backbone = str(cfg["params_backbone"])
    batch_size = int(float(cfg["params_batch_size"]))
    center_crop_key = str(cfg["params_center_crop_key"])
    coreset_sampling_ratio = float(cfg["params_coreset_sampling_ratio"])
    image_size_key = str(cfg["params_image_size_key"])
    layers_key = str(cfg["params_layers_key"])
    max_patches_per_image = int(float(cfg["params_max_patches_per_image"]))
    num_neighbors = int(float(cfg["params_num_neighbors"]))
    pre_trained = _as_bool(cfg["params_pre_trained"])
    reduction = str(cfg["params_reduction"])
    soft_corruption_level = str(cfg["params_soft_corruption_level"])
    soft_review_budget = float(cfg["params_soft_review_budget"])
    soft_train_fraction = float(cfg["params_soft_train_fraction"])
    if image_size_key not in IMAGE_SIZE_MAP:
        raise ValueError(f"Unsupported image_size_key: {image_size_key}")
    if layers_key not in LAYERS_MAP:
        raise ValueError(f"Unsupported layers_key: {layers_key}")
    image_size = IMAGE_SIZE_MAP[image_size_key]
    layers = LAYERS_MAP[layers_key]
    if center_crop_key == "none":
        center_crop_size = None
    elif center_crop_key == "0.875":
        s = image_size[0]
        center_crop_size = (int(0.875 * s), int(0.875 * s))
    else:
        raise ValueError(f"Unsupported center_crop_key: {center_crop_key}")
    return {
        "backbone": backbone, "batch_size": batch_size,
        "center_crop_size": center_crop_size,
        "coreset_sampling_ratio": coreset_sampling_ratio,
        "image_size": image_size, "layers": layers,
        "max_patches_per_image": max_patches_per_image,
        "num_neighbors": num_neighbors, "pre_trained": pre_trained,
        "reduction": reduction,
        "soft_corruption_level": soft_corruption_level,
        "soft_review_budget": soft_review_budget,
        "soft_train_fraction": soft_train_fraction,
    }


def run_patchcore_capture_scores(cfg, seed_offset=0):
    # Same protocol as notebook 5, but returns raw scores instead of only metrics.
    decoded = decode_patchcore_config(cfg)
    frac = decoded["soft_train_fraction"]
    if frac >= 1.0:
        df_sub = df_train_good.copy()
    else:
        df_sub = df_train_good.sample(frac=frac, random_state=SEED + seed_offset)
        df_sub = df_sub.reset_index(drop=True)

    train_loader, val_loader, test_loader = make_loaders(
        df_train_good_sub=df_sub,
        image_size=decoded["image_size"],
        center_crop_size=decoded["center_crop_size"],
        batch_size=decoded["batch_size"],
        corruption_level=decoded["soft_corruption_level"],
    )

    t0 = time.time()
    extractor = FeatureExtractor(
        backbone=decoded["backbone"], layers=decoded["layers"],
        pre_trained=decoded["pre_trained"],
    )
    memory_bank = fit_memory_bank(
        extractor, train_loader,
        coreset_sampling_ratio=decoded["coreset_sampling_ratio"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )
    y_true_val, y_score_val = score_images(
        extractor, val_loader, memory_bank,
        num_neighbors=decoded["num_neighbors"], reduction=decoded["reduction"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )
    y_true_test, y_score_test = score_images(
        extractor, test_loader, memory_bank,
        num_neighbors=decoded["num_neighbors"], reduction=decoded["reduction"],
        max_patches_per_image=decoded["max_patches_per_image"],
    )
    runtime_sec = time.time() - t0

    del extractor, memory_bank
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "y_true_val": y_true_val, "y_score_val": y_score_val,
        "y_true_test": y_true_test, "y_score_test": y_score_test,
        "budget": decoded["soft_review_budget"],
        "runtime_sec": runtime_sec,
    }

In [ ]:
# ============================================================
# B3b) Load MVTec per-class train/validation/test splits
# ============================================================

CLASSES = ["bottle", "cable", "capsule", "hazelnut"]


def resolve_path_with_context(path_value, csv_parent_dir):
    p = Path(str(path_value))
    if p.exists():
        return str(p.resolve())
    candidates = [
        csv_parent_dir / p, SPLITS_DIR / p, ROOT / p, ROOT.parent / p,
        ROOT / "mvtec_splits" / p, ROOT.parent / "hpo_mvtec_4class" / p,
    ]
    for c in candidates:
        if c.exists():
            return str(c.resolve())
    return str(path_value)


def load_class_split(class_name, split_filename, split_name):
    class_dir = SPLITS_DIR / class_name
    csv_path = class_dir / split_filename
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing split file: {csv_path}")
    d = pd.read_csv(csv_path)
    if "path" not in d.columns:
        raise ValueError(f"{csv_path} is missing required column: path")
    if "class_name" not in d.columns:
        d["class_name"] = class_name
    d["split"] = split_name
    if "label" not in d.columns:
        if split_name == "train_good":
            d["label"] = 0
        else:
            raise ValueError(f"{csv_path} is missing required column: label")
    d["label"] = pd.to_numeric(d["label"], errors="coerce").astype(int)
    d["path"] = d["path"].apply(lambda x: resolve_path_with_context(x, class_dir))
    return d


train_parts, val_parts, final_parts = [], [], []
for cls in CLASSES:
    train_parts.append(load_class_split(cls, "df_train_good.csv", "train_good"))
    val_parts.append(load_class_split(cls, "df_val.csv", "val"))
    final_parts.append(load_class_split(cls, "df_final.csv", "final_test"))

df_train_good = pd.concat(train_parts, ignore_index=True)
df_val = pd.concat(val_parts, ignore_index=True)
df_final = pd.concat(final_parts, ignore_index=True)

# Class arrays aligned with loader order (shuffle=False everywhere), used for
# the per-category breakdown in B5.
VAL_CLASS = df_val["class_name"].astype(str).to_numpy()
TEST_CLASS = df_final["class_name"].astype(str).to_numpy()

print("train_good:", df_train_good.shape)
print("val:", df_val.shape)
print("test:", df_final.shape)
for name, d in [("val", df_val), ("test", df_final)]:
    rate = d["path"].map(lambda x: Path(x).exists()).mean()
    print(f"{name} path_exists_rate: {rate:.3f}")
    if rate < 1.0:
        raise FileNotFoundError(f"Some image paths do not exist in the {name} split.")

In [ ]:
# ============================================================
# B3c) Capture scores for every confirmed candidate (cached)
# ============================================================

RUN_SCORE_CAPTURE = True
MAX_SCORE_CAPTURE = None  # set to e.g. 3 for a smoke test

capture_index_rows = []

cand_table = cfs.dropna(subset=["cf_id"]).copy()
cand_table["cf_id"] = cand_table["cf_id"].astype(int)
if MAX_SCORE_CAPTURE is not None:
    cand_table = cand_table.head(MAX_SCORE_CAPTURE)

for i, (_, r) in enumerate(cand_table.iterrows(), start=1):
    cf_id = int(r["cf_id"])
    npz_path = SCORES_DIR / f"cf_{cf_id}.npz"

    if npz_path.exists():
        print(f"[{i}/{len(cand_table)}] CF {cf_id}: cached, skipping.")
    elif not RUN_SCORE_CAPTURE:
        print(f"[{i}/{len(cand_table)}] CF {cf_id}: capture disabled, skipping.")
        continue
    else:
        print(f"[{i}/{len(cand_table)}] CF {cf_id}: running PatchCore with score capture ...")
        cfg = r[FEATURES].to_dict()
        try:
            out = run_patchcore_capture_scores(cfg, seed_offset=cf_id)
            np.savez_compressed(
                npz_path,
                y_true_val=out["y_true_val"], y_score_val=out["y_score_val"],
                y_true_test=out["y_true_test"], y_score_test=out["y_score_test"],
                budget=np.array([out["budget"]]),
                runtime_sec=np.array([out["runtime_sec"]]),
            )
            print(f"  saved {npz_path.name} (runtime {out['runtime_sec']:.1f}s)")
        except Exception as e:
            print(f"  FAILED CF {cf_id}: {repr(e)}")
            continue

    # Sanity: recompute the confirmed F1 from the captured scores.
    z = np.load(npz_path)
    thr = threshold_from_review_budget(z["y_score_val"], float(z["budget"][0]))
    y_pred_test = (z["y_score_test"] >= thr).astype(int)
    _, _, f1_rec, _ = precision_recall_fscore_support(
        z["y_true_test"], y_pred_test, average="binary", zero_division=0
    )
    try:
        auroc_rec = roc_auc_score(z["y_true_test"], z["y_score_test"])
    except ValueError:
        auroc_rec = np.nan

    prev_row = ok_df[ok_df["cf_id"] == cf_id] if not ok_df.empty else pd.DataFrame()
    f1_prev = float(prev_row["f1_test"].iloc[0]) if len(prev_row) else np.nan
    capture_index_rows.append({
        "cf_id": cf_id,
        "method": str(r.get("method", "Unknown")),
        "budget": float(z["budget"][0]),
        "f1_test_recomputed": float(f1_rec),
        "f1_test_prior": f1_prev,
        "f1_abs_diff_vs_prior": abs(float(f1_rec) - f1_prev) if pd.notna(f1_prev) else np.nan,
        "auroc_test_recomputed": float(auroc_rec),
        "npz": str(npz_path),
    })

capture_index = pd.DataFrame(capture_index_rows)
capture_index_path = REV_DIR / "score_capture_index.csv"
capture_index.to_csv(capture_index_path, index=False)
print("\nSaved capture index:", capture_index_path)
display(capture_index.round(4))

if not capture_index.empty and capture_index["f1_abs_diff_vs_prior"].notna().any():
    print("Max |recomputed - prior| confirmed F1:",
          f"{capture_index['f1_abs_diff_vs_prior'].max():.4f}",
          "(small values confirm the reproduction matches notebook 5).")

In [ ]:
# ============================================================
# B4) Review-budget sweep and threshold-transfer loss
# ============================================================

from sklearn.metrics import f1_score

BUDGETS = np.round(np.arange(0.005, 0.5001, 0.005), 3)

sweep_rows = []
curves = {}

npz_files = sorted(SCORES_DIR.glob("cf_*.npz"))
if not npz_files:
    raise ValueError("No captured scores found. Run cell B3c first.")

for npz_path in npz_files:
    cf_id = int(npz_path.stem.split("_")[1])
    z = np.load(npz_path)
    y_true_val, y_score_val = z["y_true_val"], z["y_score_val"]
    y_true_test, y_score_test = z["y_true_test"], z["y_score_test"]
    own_budget = float(z["budget"][0])

    f1_curve = []
    for b in BUDGETS:
        thr = threshold_from_review_budget(y_score_val, float(b))
        f1_b = f1_score(y_true_test, (y_score_test >= thr).astype(int), zero_division=0)
        f1_curve.append(f1_b)
    f1_curve = np.asarray(f1_curve)
    curves[cf_id] = f1_curve

    thr_own = threshold_from_review_budget(y_score_val, own_budget)
    f1_own = f1_score(y_true_test, (y_score_test >= thr_own).astype(int), zero_division=0)

    # Oracle: best F1 over all test-score thresholds (upper bound with these scores).
    order = np.sort(np.unique(y_score_test))
    best_f1_oracle = 0.0
    for thr_t in order:
        f1_t = f1_score(y_true_test, (y_score_test >= thr_t).astype(int), zero_division=0)
        best_f1_oracle = max(best_f1_oracle, f1_t)

    method = "Unknown"
    row_m = cfs[cfs["cf_id"] == cf_id]
    if len(row_m) and "method" in row_m.columns:
        method = str(row_m["method"].iloc[0])

    sweep_rows.append({
        "cf_id": cf_id,
        "method": method,
        "own_budget": own_budget,
        "f1_at_own_budget": f1_own,
        "f1_best_over_budget_sweep": float(f1_curve.max()),
        "budget_at_best_sweep_f1": float(BUDGETS[int(f1_curve.argmax())]),
        "f1_oracle_threshold": best_f1_oracle,
        "threshold_transfer_loss": best_f1_oracle - f1_own,
    })

sweep_table = pd.DataFrame(sweep_rows).sort_values("threshold_transfer_loss", ascending=False)
display(sweep_table.round(4))
save_table(sweep_table.round(4), "mvtec_budget_sweep_summary")

print("Mean threshold-transfer loss (oracle F1 - F1 at own budget): "
      f"{sweep_table['threshold_transfer_loss'].mean():.4f}")

# Figure: per-method mean budget-sweep curve + each candidate's own operating point.
fig, ax = plt.subplots(figsize=SINGLE_COL)
for m in [m for m in METHOD_ORDER if m in sweep_table["method"].unique()]:
    ids = sweep_table.loc[sweep_table["method"] == m, "cf_id"].tolist()
    mat = np.vstack([curves[j] for j in ids])
    mean_curve = mat.mean(axis=0)
    lo = mat.min(axis=0)
    hi = mat.max(axis=0)
    color = METHOD_COLORS.get(m)
    ax.plot(BUDGETS, mean_curve, label=m, color=color, linewidth=1.1)
    ax.fill_between(BUDGETS, lo, hi, color=color, alpha=0.12, linewidth=0)
    sub = sweep_table[sweep_table["method"] == m]
    ax.scatter(sub["own_budget"], sub["f1_at_own_budget"], s=16,
               color=color, marker=METHOD_MARKERS.get(m, "o"),
               edgecolor="black", linewidth=0.3, zorder=5)

ax.set_xlabel("Review budget")
ax.set_ylabel("Confirmed test F1")
ax.grid(linewidth=0.4, alpha=0.35)
ax.legend(frameon=False, loc="best", title=None)
save_paper_fig(fig, "mvtec_budget_sweep_f1")

### B5 — Per-category breakdown (MVTec): does pooled thresholding explain the near-chance AUROC?

The paper pools four MVTec categories and applies a single validation-calibrated budget
threshold. If anomaly-score *scales* differ across categories, the pooled ranking can be near
chance even when every within-category ranking is good. The cell below computes, from the
captured scores:

- pooled test AUROC (as in the paper) vs. the mean **per-category** test AUROC;
- per-category F1 under the pooled threshold vs. under a **per-category** threshold at the same
  review budget.

A large pooled-vs-per-category AUROC difference identifies score-scale mismatch under pooled
thresholding — not surrogate error alone — as a concrete mechanism behind the MVTec confirmation
gap, which is exactly the diagnosis the review requests.

In [ ]:
# ============================================================
# B5) Per-category AUROC and F1 from captured scores (MVTec)
# ============================================================

percat_rows = []

for npz_path in sorted(SCORES_DIR.glob("cf_*.npz")):
    cf_id = int(npz_path.stem.split("_")[1])
    z = np.load(npz_path)
    y_true_val, y_score_val = z["y_true_val"], z["y_score_val"]
    y_true_test, y_score_test = z["y_true_test"], z["y_score_test"]
    own_budget = float(z["budget"][0])

    if len(y_true_val) != len(VAL_CLASS) or len(y_true_test) != len(TEST_CLASS):
        print(f"CF {cf_id}: captured score length does not match split length; skipping "
              "(splits changed since capture?).")
        continue

    method = "Unknown"
    row_m = cfs[cfs["cf_id"] == cf_id]
    if len(row_m) and "method" in row_m.columns:
        method = str(row_m["method"].iloc[0])

    # Pooled thresholding (paper protocol).
    thr_pooled = threshold_from_review_budget(y_score_val, own_budget)
    y_pred_pooled = (y_score_test >= thr_pooled).astype(int)

    try:
        auroc_pooled = roc_auc_score(y_true_test, y_score_test)
    except ValueError:
        auroc_pooled = np.nan

    for cls in sorted(np.unique(TEST_CLASS)):
        m_test = TEST_CLASS == cls
        m_val = VAL_CLASS == cls
        yt, ys = y_true_test[m_test], y_score_test[m_test]

        try:
            auroc_cls = roc_auc_score(yt, ys)
        except ValueError:
            auroc_cls = np.nan

        f1_pooled_cls = f1_score(yt, y_pred_pooled[m_test], zero_division=0)

        thr_cls = threshold_from_review_budget(y_score_val[m_val], own_budget)
        f1_classwise = f1_score(yt, (ys >= thr_cls).astype(int), zero_division=0)

        percat_rows.append({
            "cf_id": cf_id, "method": method, "class_name": cls,
            "budget": own_budget,
            "auroc_pooled": auroc_pooled,
            "auroc_class": auroc_cls,
            "f1_class_pooled_threshold": f1_pooled_cls,
            "f1_class_classwise_threshold": f1_classwise,
        })

percat = pd.DataFrame(percat_rows)
if percat.empty:
    raise ValueError("No per-category rows; check score capture and split alignment.")

percat_path = REV_DIR / "mvtec_per_category_breakdown.csv"
percat.to_csv(percat_path, index=False)
print("Saved per-category breakdown:", percat_path)

# Candidate-level comparison: pooled AUROC vs mean per-category AUROC,
# and F1 gain from class-wise thresholding.
cand_level = (
    percat.groupby(["cf_id", "method"], as_index=False)
    .agg(
        auroc_pooled=("auroc_pooled", "first"),
        auroc_per_class_mean=("auroc_class", "mean"),
        f1_pooled_thr_mean=("f1_class_pooled_threshold", "mean"),
        f1_classwise_thr_mean=("f1_class_classwise_threshold", "mean"),
    )
)
cand_level["auroc_pooling_penalty"] = (
    cand_level["auroc_per_class_mean"] - cand_level["auroc_pooled"]
)
cand_level["f1_gain_classwise_threshold"] = (
    cand_level["f1_classwise_thr_mean"] - cand_level["f1_pooled_thr_mean"]
)
display(cand_level.round(4))
save_table(cand_level.round(4), "mvtec_pooled_vs_perclass_auroc")

print(f"\nMean pooled test AUROC:            {cand_level['auroc_pooled'].mean():.4f}")
print(f"Mean per-category test AUROC:      {cand_level['auroc_per_class_mean'].mean():.4f}")
print(f"Mean AUROC pooling penalty:        {cand_level['auroc_pooling_penalty'].mean():.4f}")
print(f"Mean F1 gain, class-wise threshold: {cand_level['f1_gain_classwise_threshold'].mean():.4f}")
print("A large positive pooling penalty indicates score-scale mismatch across categories "
      "under pooled thresholding as the mechanism, rather than surrogate error alone.")

# Figure 1: pooled vs per-category AUROC per method.
fig, ax = plt.subplots(figsize=SINGLE_COL)
methods_present = [m for m in METHOD_ORDER if m in cand_level["method"].unique()]
x = np.arange(len(methods_present))
width = 0.35
pooled_means = [cand_level.loc[cand_level["method"] == m, "auroc_pooled"].mean()
                for m in methods_present]
percls_means = [cand_level.loc[cand_level["method"] == m, "auroc_per_class_mean"].mean()
                for m in methods_present]
ax.bar(x - width / 2, pooled_means, width, label="Pooled AUROC", color="tab:red", alpha=0.85)
ax.bar(x + width / 2, percls_means, width, label="Mean per-category AUROC",
       color="tab:blue", alpha=0.85)
ax.axhline(0.5, linewidth=0.7, linestyle="--", color="black", alpha=0.6)
ax.text(len(methods_present) - 0.5, 0.505, "chance", fontsize=6, ha="right")
ax.set_xticks(x)
ax.set_xticklabels(methods_present)
ax.set_ylabel("Test AUROC")
ax.set_ylim(0.3, 1.0)
ax.grid(axis="y", linewidth=0.4, alpha=0.35)
ax.legend(frameon=False, loc="lower right")
save_paper_fig(fig, "mvtec_pooled_vs_perclass_auroc")

# Figure 2: heatmap of per-category AUROC across candidates.
heat = percat.pivot_table(index="cf_id", columns="class_name", values="auroc_class")
fig, ax = plt.subplots(figsize=(3.35, 3.1))
im = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn", vmin=0.3, vmax=1.0)
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=30, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([f"CF {i}" for i in heat.index], fontsize=5)
ax.set_xlabel("MVTec category")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Per-category test AUROC", fontsize=7)
save_paper_fig(fig, "mvtec_perclass_auroc_heatmap")

# Extend the manifest inputs (picked up by the final cell via saved_* lists).
print("\nPer-category diagnosis complete.")

## Part C — Multi-factual sensitivity of the generator comparison (review comment M4)

The paper's generator comparison used a single factual configuration per dataset. The review
asks whether the reported generator ranking is robust to the choice of factual anchor. The cells
below repeat the surrogate-level comparison for **all** the percentile-based factual anchors
already defined in notebook 4 (trials nearest the 50th/70th/85th/95th percentiles of validation
F1, plus the best trial), under the identical paper query: `target_band` objective with bounds
`[0.75, 0.95]` and free soft controls, `K = 8` candidates per generator.

Surrogate-level metrics (validity, proximity, runtime, surrogate F1) are cheap and computed for
every anchor. Real-model confirmation of the top candidate per generator per anchor is optional
(`CONFIRM_TOP_PER_ANCHOR`), reusing the PatchCore stack from Part B.

ACFX note: the ACFX adapter from notebook 4 is environment-dependent. If `acfx` is not
importable here, assign your adapter function to `ACFX_GENERATOR` (signature:
`fn(x_factual_df, desired_range, permitted_range, k) -> DataFrame of candidate configurations`)
or the comparison runs with DiCE and CFNOW only, which is reported transparently.

In [ ]:
# ============================================================
# C1) Generation helpers (validation, distance, ranking) — same
#     semantics as notebook 4
# ============================================================

def domain_for_feature(df_ref, feat):
    if feat in CATEGORICAL:
        return sorted(df_ref[feat].dropna().astype(str).unique().tolist())
    col = pd.to_numeric(df_ref[feat], errors="coerce")
    return [float(np.nanmin(col)), float(np.nanmax(col))]


def mixed_distance(row, factual_row, df_ref):
    distance = 0.0
    for c in NUMERIC:
        x = float(row[c])
        x0 = float(factual_row[c])
        dom_lo, dom_hi = domain_for_feature(df_ref, c)
        span = max(dom_hi - dom_lo, 1e-9)
        distance += abs(x - x0) / span
    for c in CATEGORICAL:
        distance += int(str(row[c]) != str(factual_row[c]))
    return float(distance)


def prepare_candidate_table(candidates, method_name, factual_row, df_ref,
                            desired_range, model):
    # Soft controls are free in the paper query, so validity reduces to the
    # objective interval on the fresh surrogate prediction.
    if candidates is None or len(candidates) == 0:
        return pd.DataFrame()
    raw = candidates.copy().reset_index(drop=True)
    extra = [c for c in raw.columns if c not in FEATURES]
    out = raw.drop(columns=extra, errors="ignore").reindex(columns=FEATURES)
    out = coerce_schema(out).dropna(subset=FEATURES).reset_index(drop=True)
    if out.empty:
        return out
    out["surrogate_pred"] = predict_surrogate(model, out)
    L, U = desired_range
    out["strictly_valid"] = (out["surrogate_pred"] >= L) & (out["surrogate_pred"] <= U)
    out["distance_to_factual"] = [
        mixed_distance(row, factual_row, df_ref) for _, row in out.iterrows()
    ]
    out["method"] = method_name
    out = out.sort_values(
        by=["strictly_valid", "surrogate_pred", "distance_to_factual"],
        ascending=[False, False, True],
    ).reset_index(drop=True)
    return out


def select_top_k(common_df, k):
    if common_df is None or common_df.empty:
        return pd.DataFrame()
    strict = common_df[common_df["strictly_valid"]].copy()
    if len(strict) >= k:
        return strict.head(k).reset_index(drop=True)
    print(f"Only {len(strict)} strictly valid candidates; returning top {k} ranked.")
    return common_df.head(k).copy().reset_index(drop=True)


# Paper query (identical on both datasets in the technical evaluation).
DESIRED_RANGE = [0.75, 0.95]
K_RETURN = 8

# Free soft controls: permitted ranges are the full observed domains.
PERMITTED_RANGE = {feat: domain_for_feature(df, feat) for feat in FEATURES}
print("Objective: target_band", DESIRED_RANGE, "| K =", K_RETURN, "| soft controls free")

In [ ]:
# ============================================================
# C2) Factual anchors: percentiles of the HPO-log target + best trial
# ============================================================

ANCHOR_QUANTILES = [0.50, 0.70, 0.85, 0.95]

anchor_indices = []
for q in ANCHOR_QUANTILES:
    tv = df[TARGET].quantile(q)
    anchor_indices.append(int((df[TARGET] - tv).abs().idxmin()))
anchor_indices.append(int(df[TARGET].idxmax()))
anchor_indices = list(dict.fromkeys(anchor_indices))

anchor_labels = [f"p{int(q*100)}" for q in ANCHOR_QUANTILES] + ["best"]
anchor_labels = anchor_labels[: len(anchor_indices)]

anchors = []
for label, idx in zip(anchor_labels, anchor_indices):
    row = df.loc[idx, FEATURES + [TARGET]].copy()
    x_df = coerce_schema(pd.DataFrame([row[FEATURES].values], columns=FEATURES))
    pred = float(predict_surrogate(cb, x_df)[0])
    anchors.append({
        "label": label,
        "index": idx,
        "observed": float(row[TARGET]),
        "surrogate": pred,
        "x_factual": row[FEATURES].copy(),
        "x_factual_df": x_df,
    })

anchor_table = pd.DataFrame([
    {
        "anchor": a["label"], "log_index": a["index"],
        "observed_F1": a["observed"], "surrogate_F1": a["surrogate"],
        "review_budget": float(a["x_factual"]["params_soft_review_budget"]),
        "train_fraction": float(a["x_factual"]["params_soft_train_fraction"]),
        "corruption": str(a["x_factual"]["params_soft_corruption_level"]),
    }
    for a in anchors
])
display(anchor_table.round(4))
save_table(anchor_table.round(4), "mvtec_multifactual_anchors")

In [ ]:
# ============================================================
# C3) Run the generators for every factual anchor
# ============================================================

import dice_ml
from dice_ml import Dice

# DiCE setup (once).
df_dice = df[FEATURES + [TARGET]].copy()
for c in CATEGORICAL:
    df_dice[c] = df_dice[c].astype(str)
CONTINUOUS = [c for c in FEATURES if c not in CATEGORICAL]

data_dice = dice_ml.Data(dataframe=df_dice, continuous_features=CONTINUOUS,
                         outcome_name=TARGET)
model_dice = dice_ml.Model(model=cb, backend="sklearn", model_type="regressor")
exp_dice = Dice(data_dice, model_dice, method="random")


def generate_dice(x_factual_df, k):
    pool = int(max(30, 10 * k))
    t0 = time.time()
    res = exp_dice.generate_counterfactuals(
        x_factual_df[FEATURES],
        total_CFs=pool,
        desired_range=DESIRED_RANGE,
        features_to_vary=FEATURES,
        permitted_range=PERMITTED_RANGE,
    )
    runtime = time.time() - t0
    ex = res.cf_examples_list[0]
    if hasattr(ex, "final_cfs_df_sparse") and ex.final_cfs_df_sparse is not None:
        raw = ex.final_cfs_df_sparse.copy()
    else:
        raw = ex.final_cfs_df.copy()
    return raw, runtime


def generate_cfnow(k):
    # CFNOW-style search over the observed HPO log (as in notebook 4).
    t0 = time.time()
    raw = df[FEATURES].copy()
    return raw, time.time() - t0


# Optional ACFX hook: assign your notebook-4 adapter here if available.
ACFX_GENERATOR = None
try:
    import acfx  # noqa: F401
    print("acfx importable; assign ACFX_GENERATOR to include it in the comparison.")
except Exception:
    print("acfx not importable in this kernel; comparison runs with DiCE and CFNOW.")


multi_rows = []
multi_candidates = []

for a in anchors:
    label = a["label"]
    print("\n" + "=" * 70)
    print(f"Anchor {label} (log index {a['index']}, surrogate {a['surrogate']:.4f})")
    print("=" * 70)

    generators = [("DiCE", lambda: generate_dice(a["x_factual_df"], K_RETURN)),
                  ("CFNOW", lambda: generate_cfnow(K_RETURN))]
    if ACFX_GENERATOR is not None:
        generators.append((
            "ACFX",
            lambda: (ACFX_GENERATOR(a["x_factual_df"], DESIRED_RANGE,
                                    PERMITTED_RANGE, K_RETURN), np.nan),
        ))

    for method, fn in generators:
        try:
            raw, runtime = fn()
        except Exception as e:
            print(f"{method} failed on anchor {label}: {repr(e)}")
            continue

        common = prepare_candidate_table(
            candidates=raw, method_name=method, factual_row=a["x_factual"],
            df_ref=df, desired_range=DESIRED_RANGE, model=cb,
        )
        top = select_top_k(common, K_RETURN)
        if top.empty:
            print(f"{method}: no candidates on anchor {label}")
            continue

        top["anchor"] = label
        top["anchor_index"] = a["index"]
        top["generation_sec"] = runtime
        multi_candidates.append(top)

        multi_rows.append({
            "anchor": label,
            "method": method,
            "returned": int(len(top)),
            "strictly_valid": int(top["strictly_valid"].sum()),
            "strictly_valid_pct": float(100 * top["strictly_valid"].mean()),
            "mean_surrogate_f1": float(top["surrogate_pred"].mean()),
            "mean_distance": float(top["distance_to_factual"].mean()),
            "generation_sec": float(runtime) if pd.notna(runtime) else np.nan,
        })
        print(f"{method}: valid {int(top['strictly_valid'].sum())}/{len(top)}, "
              f"mean surrogate F1 {top['surrogate_pred'].mean():.4f}, "
              f"mean distance {top['distance_to_factual'].mean():.3f}, "
              f"gen time {runtime:.2f}s")

multi_summary = pd.DataFrame(multi_rows)
multi_all = (pd.concat(multi_candidates, ignore_index=True)
             if multi_candidates else pd.DataFrame())

display(multi_summary.round(4))
save_table(multi_summary.round(4), "mvtec_multifactual_generator_summary")

multi_all_path = REV_DIR / "mvtec_multifactual_candidates.csv"
multi_all.to_csv(multi_all_path, index=False)
print("Saved all multi-factual candidates:", multi_all_path)

In [ ]:
# ============================================================
# C4) Multi-factual figures: validity, proximity, surrogate F1
# ============================================================

if multi_summary.empty:
    raise ValueError("No multi-factual results; run cell C3 first.")

methods_present = [m for m in METHOD_ORDER if m in multi_summary["method"].unique()]
anchors_present = [a["label"] for a in anchors if a["label"] in multi_summary["anchor"].unique()]
x = np.arange(len(anchors_present))
width = 0.8 / max(len(methods_present), 1)

fig, axes = plt.subplots(1, 3, figsize=WIDE_COL)
panels = [
    ("strictly_valid_pct", "Strict validity (%)", axes[0]),
    ("mean_distance", "Mean distance to factual", axes[1]),
    ("mean_surrogate_f1", "Mean surrogate F1", axes[2]),
]
for col, ylabel, ax in panels:
    for j, m in enumerate(methods_present):
        sub = multi_summary[multi_summary["method"] == m].set_index("anchor")
        vals = [sub.loc[a, col] if a in sub.index else np.nan for a in anchors_present]
        ax.bar(x + (j - (len(methods_present) - 1) / 2) * width, vals,
               width=width * 0.95, label=m, color=METHOD_COLORS.get(m), alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(anchors_present)
    ax.set_xlabel("Factual anchor")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", linewidth=0.4, alpha=0.35)

axes[0].legend(frameon=False, loc="lower right")
fig.tight_layout()
save_paper_fig(fig, "mvtec_multifactual_generator_comparison")

# Ranking stability: method rank by mean surrogate F1 per anchor.
rank_table = (
    multi_summary
    .pivot(index="anchor", columns="method", values="mean_surrogate_f1")
    .reindex(anchors_present)
)
rank_of = rank_table.rank(axis=1, ascending=False).astype("Int64")
print("Method rank by mean surrogate F1 (1 = best) per anchor:")
display(rank_of)
rank_out = rank_of.reset_index()
save_table(rank_out, "mvtec_multifactual_ranking_stability", float_format="%d")

In [ ]:
# ============================================================
# C5) Optional: real-model confirmation of the top candidate
#     per generator per anchor
# ============================================================

CONFIRM_TOP_PER_ANCHOR = False  # set True to spend up to (anchors x methods) retrainings

if CONFIRM_TOP_PER_ANCHOR and not multi_all.empty:
    confirm_rows = []
    top_per = (
        multi_all[multi_all["strictly_valid"]]
        .sort_values(["anchor", "method", "surrogate_pred"], ascending=[True, True, False])
        .groupby(["anchor", "method"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )
    print(f"Confirming {len(top_per)} candidates with real PatchCore ...")

    for i, (_, r) in enumerate(top_per.iterrows(), start=1):
        tag = f"{r['anchor']}_{r['method']}"
        print(f"\n[{i}/{len(top_per)}] {tag}")
        try:
            out = run_patchcore_capture_scores(r[FEATURES].to_dict(),
                                               seed_offset=1000 + i)
            thr = threshold_from_review_budget(out["y_score_val"], out["budget"])
            y_pred = (out["y_score_test"] >= thr).astype(int)
            _, _, f1_t, _ = precision_recall_fscore_support(
                out["y_true_test"], y_pred, average="binary", zero_division=0
            )
            try:
                auroc_t = roc_auc_score(out["y_true_test"], out["y_score_test"])
            except ValueError:
                auroc_t = np.nan
            confirm_rows.append({
                "anchor": r["anchor"], "method": r["method"],
                "surrogate_f1": float(r["surrogate_pred"]),
                "confirmed_f1": float(f1_t),
                "confirmed_auroc": float(auroc_t),
                "gap": float(f1_t) - float(r["surrogate_pred"]),
                "distance_to_factual": float(r["distance_to_factual"]),
                "runtime_sec": float(out["runtime_sec"]),
            })
            print(f"  surrogate {r['surrogate_pred']:.4f} -> confirmed {f1_t:.4f} "
                  f"(gap {f1_t - r['surrogate_pred']:+.4f})")
        except Exception as e:
            print(f"  FAILED: {repr(e)}")

    confirm_table = pd.DataFrame(confirm_rows)
    display(confirm_table.round(4))
    save_table(confirm_table.round(4), "mvtec_multifactual_confirmation")

    if not confirm_table.empty:
        fig, ax = plt.subplots(figsize=SINGLE_COL)
        anchors_c = [a["label"] for a in anchors
                     if a["label"] in confirm_table["anchor"].unique()]
        for m in [m for m in METHOD_ORDER if m in confirm_table["method"].unique()]:
            sub = confirm_table[confirm_table["method"] == m].set_index("anchor")
            vals = [sub.loc[a, "gap"] if a in sub.index else np.nan for a in anchors_c]
            ax.plot(range(len(anchors_c)), vals, marker=METHOD_MARKERS.get(m, "o"),
                    color=METHOD_COLORS.get(m), label=m, linewidth=1.0)
        ax.axhline(0.0, linewidth=0.6, color="black", alpha=0.5)
        ax.set_xticks(range(len(anchors_c)))
        ax.set_xticklabels(anchors_c)
        ax.set_xlabel("Factual anchor")
        ax.set_ylabel("Confirmation gap")
        ax.grid(linewidth=0.4, alpha=0.35)
        ax.legend(frameon=False, loc="best")
        save_paper_fig(fig, "mvtec_multifactual_confirmation_gap")
else:
    print("CONFIRM_TOP_PER_ANCHOR is False; surrogate-level sensitivity only "
          "(enable to add real-model evidence for the revised paper).")

In [ ]:
# ============================================================
# Final: revision manifest (maps outputs to review comments)
# ============================================================

manifest = {
    "dataset": "mvtec",
    "notebook": "revision_analysis",
    "review_comment_mapping": {
        "M1_compute_sustainability": {
            "tables": [
                str(TABLE_DIR / "mvtec_compute_cost_per_query.csv"),
                str(TABLE_DIR / "mvtec_compute_cost_workloads.csv"),
            ],
            "figures": [str(FIG_DIR / "mvtec_compute_cost_per_query.pdf")],
        },
        "M2_confirmation_gap_diagnosis": {
            "tables": [
                str(TABLE_DIR / "mvtec_gap_vs_distance_correlations.csv"),
                str(TABLE_DIR / "mvtec_worst_gap_candidates.csv"),
                str(TABLE_DIR / "mvtec_degeneracy_flag_summary.csv"),
                str(TABLE_DIR / "mvtec_budget_sweep_summary.csv"),
            ],
            "figures": [
                str(FIG_DIR / "mvtec_gap_vs_distance.pdf"),
                str(FIG_DIR / "mvtec_gap_vs_degeneracy_flags.pdf"),
                str(FIG_DIR / "mvtec_budget_sweep_f1.pdf"),
            ],
            "captured_scores_dir": str(SCORES_DIR),
        },
        "M4_multifactual_sensitivity": {
            "tables": [
                str(TABLE_DIR / "mvtec_multifactual_anchors.csv"),
                str(TABLE_DIR / "mvtec_multifactual_generator_summary.csv"),
                str(TABLE_DIR / "mvtec_multifactual_ranking_stability.csv"),
            ],
            "figures": [str(FIG_DIR / "mvtec_multifactual_generator_comparison.pdf")],
            "candidates_csv": str(REV_DIR / "mvtec_multifactual_candidates.csv"),
        },
    },
    "assumptions": {
        "patchcore_power_w": PATCHCORE_POWER_W,
        "surrogate_power_w": SURROGATE_POWER_W,
        "grid_kgco2_per_kwh": GRID_KGCO2_PER_KWH,
        "paper_query": {"objective": "target_band", "bounds": DESIRED_RANGE,
                        "K": K_RETURN, "soft_controls": "free"},
        "anchor_quantiles": ANCHOR_QUANTILES,
        "seed": SEED,
    },
    "all_saved_figures": [str(p) for p in saved_figures],
    "all_saved_tables": [str(p) for p in saved_tables],
}

manifest_path = REV_DIR / "revision_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved revision manifest:", manifest_path)
print(f"\n{len(saved_figures)} figure files and {len(saved_tables)} table files produced.")
print("\nDone. Outputs to cite in the revised manuscript:")
for k, v in manifest["review_comment_mapping"].items():
    print(f"- {k}")
    for t in v.get("tables", []):
        print("    table:", Path(t).name)
    for g in v.get("figures", []):
        print("    figure:", Path(g).name)